# 04 — Edge Deployment: ONNX Export, Optimization, and Inference Benchmarking

**Covers:**
- `notes/02-advanced-deep-learning/ch11-deployment/` — model export pipeline
- `notes/02-advanced-deep-learning/ch10-pruning-mixed-precision/` — inference optimization

**Goal:** Export the compressed model from `03_compression_analysis.ipynb` to platform-neutral formats (ONNX, TFLite), validate numerical equivalence, and measure inference latency at the precision needed to confirm the ProductionCV <50ms/frame constraint.

**Target hardware:** ARM-based edge device (Jetson Nano / Raspberry Pi 4). This notebook runs the export pipeline on any machine; actual device benchmarks are logged separately.

**Prerequisites:**
- `03_compression_analysis.ipynb` completed, or pretrained checkpoint available
- `onnxruntime`, `onnx` installed (`pip install onnxruntime onnx`)
- TFLite conversion requires TensorFlow (optional — cell handles missing TF gracefully)

**`QUICK_MODE`:** Exports a tiny ResNet-18 classifier instead of the full Faster R-CNN (export completes in seconds, no large checkpoint needed).

In [ ]:
# ── Imports & load model for export ──────────────────────────────────────────
import sys, os, time, tempfile
import numpy as np
import torch
import torch.nn as nn
import torchvision
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# QUICK_MODE=True: export a ResNet-18 classifier (fast, no checkpoint needed)
# QUICK_MODE=False: export the full Faster R-CNN from notebook 03
QUICK_MODE  = False
MODEL_DIR   = Path('../models')
EXPORT_DIR  = MODEL_DIR / 'exports'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE      = 'cpu'  # ONNX export always happens on CPU
IMG_SIZE    = 224 if QUICK_MODE else 416  # input spatial resolution
NUM_CLASSES = 6

if QUICK_MODE:
    # Use a simple ResNet-18 classifier as a stand-in for quick testing
    model = torchvision.models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    DUMMY_INPUT = torch.rand(1, 3, IMG_SIZE, IMG_SIZE)
    print(f'[QUICK_MODE] Using ResNet-18 classifier ({IMG_SIZE}×{IMG_SIZE} input)')
else:
    # Load distilled/compressed model from notebook 03
    from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn
    from torchvision.models.detection import FasterRCNN_MobileNet_V3_Large_FPN_Weights
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

    model = fasterrcnn_mobilenet_v3_large_fpn(
        weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

    ckpt_path = MODEL_DIR / 'student_distilled.pth'
    if ckpt_path.exists():
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        print(f'Loaded distilled student from {ckpt_path}')
    else:
        print('[NOTE] Distilled checkpoint not found — using pretrained weights.')
        print('Run 03_compression_analysis.ipynb to train and save student_distilled.pth')
    # Detection models need a list-of-tensors as input for ONNX export
    DUMMY_INPUT = [torch.rand(3, IMG_SIZE, IMG_SIZE)]

model = model.to(DEVICE)
model.eval()
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model loaded: {total_params:.1f}M params | Export dir: {EXPORT_DIR}')

## 1. ONNX Export — Platform-Neutral Format

**Why ONNX?** Open Neural Network Exchange is a graph IR (intermediate representation) that:
- Decouples model training (PyTorch) from inference backend (TensorRT, ONNX Runtime, CoreML, TFLite)
- Enables hardware-specific optimisation without changing training code
- Is the standard format for deploying to NVIDIA Triton, Azure ML, ONNX Runtime on ARM

**`dynamic_axes`:** Specifies which dimensions can vary at runtime. Setting batch and spatial dimensions as dynamic means the exported model accepts any image size — important for retail cameras that may be different resolutions.

**`opset_version`:** ONNX evolves its operator set. 17 is a safe modern choice; Jetson Nano's TensorRT 8.x supports up to opset 17.

In [ ]:
# ── torch.onnx.export → validate with onnxruntime ────────────────────────────
try:
    import onnx
    import onnxruntime as ort
    HAS_ONNX = True
except ImportError:
    HAS_ONNX = False
    print('onnxruntime/onnx not installed. Run: pip install onnxruntime onnx')

onnx_path = EXPORT_DIR / 'model.onnx'

if HAS_ONNX:
    # Export the model to ONNX format
    # For detection models (list input), we must wrap in a thin nn.Module
    if QUICK_MODE:
        # Classifier: standard export
        torch.onnx.export(
            model,
            DUMMY_INPUT,
            str(onnx_path),
            opset_version=17,
            input_names=['image'],
            output_names=['logits'],
            dynamic_axes={
                'image':  {0: 'batch_size', 2: 'height', 3: 'width'},
                'logits': {0: 'batch_size'},
            },
            do_constant_folding=True,  # fold constant expressions → smaller graph
        )
    else:
        # Detection model: torchvision Faster R-CNN supports ONNX export
        # but requires a patched export for dynamic batch — use script mode
        print('[NOTE] Exporting Faster R-CNN detection models to ONNX requires')
        print('       a wrapper class due to the list-of-tensors input format.')
        print('       See src/edge.py for the production EdgeExporter.export_onnx() method.')
        print('       Falling back to QUICK_MODE export for demonstration...')
        demo_model = torchvision.models.resnet18(weights=None)
        demo_model.fc = nn.Linear(demo_model.fc.in_features, NUM_CLASSES)
        demo_model.eval()
        demo_input = torch.rand(1, 3, IMG_SIZE, IMG_SIZE)
        torch.onnx.export(demo_model, demo_input, str(onnx_path),
                          opset_version=17, input_names=['image'],
                          output_names=['logits'],
                          dynamic_axes={'image': {0: 'batch_size'},
                                        'logits': {0: 'batch_size'}},
                          do_constant_folding=True)

    # Validate the exported ONNX model
    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)
    onnx_size_mb = os.path.getsize(str(onnx_path)) / 1e6
    print(f'✓ ONNX model exported and validated')
    print(f'  Path: {onnx_path}')
    print(f'  Size: {onnx_size_mb:.1f} MB')
    print(f'  Nodes: {len(onnx_model.graph.node)}')
    print(f'  IR version: {onnx_model.ir_version}')

    # Numerical validation: PyTorch output vs ONNX Runtime output must be close
    test_input = DUMMY_INPUT if QUICK_MODE else torch.rand(1, 3, IMG_SIZE, IMG_SIZE)
    with torch.no_grad():
        pt_out = (demo_model if not QUICK_MODE else model)(test_input)
    session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    ort_out = session.run(None, {'image': test_input.numpy()})[0]
    max_diff = np.abs(pt_out.numpy() - ort_out).max()
    print(f'  Max output diff (PyTorch vs ONNX Runtime): {max_diff:.2e}  ', end='')
    print('✓ PASS' if max_diff < 1e-4 else '✗ FAIL — check opset or model structure')
else:
    print('[Skipped — install onnxruntime to run this cell]')

## 2. ONNX Optimization — Constant Folding and Operator Fusion

ONNX Runtime has a built-in **graph optimizer** that applies compiler-style transforms:
- **Constant folding:** Pre-compute subgraphs with no variable inputs (e.g., `weight × 1.0` → just `weight`)
- **Operator fusion:** Merge consecutive operations (Conv + BN + ReLU → single fused kernel)
- **Layout optimization:** Reorder memory layout (NCHW ↔ NHWC) to match hardware-preferred format

`ORT_ENABLE_ALL` applies all available optimizations. The optimized model is saved as a new `.onnx` file — the original is unchanged so you can benchmark both.

In [ ]:
# ── ONNX Runtime graph optimization ──────────────────────────────────────────
if HAS_ONNX and onnx_path.exists():
    optimized_path = EXPORT_DIR / 'model_optimized.onnx'

    # Configure session with full optimization enabled
    sess_options = ort.SessionOptions()
    sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    # Save the optimized model to disk (allows inspection + reuse)
    sess_options.optimized_model_filepath = str(optimized_path)

    # Creating the session triggers optimization and saves the result
    opt_session = ort.InferenceSession(
        str(onnx_path),
        sess_options=sess_options,
        providers=['CPUExecutionProvider'],
    )

    orig_size = os.path.getsize(str(onnx_path)) / 1e6
    opt_size  = os.path.getsize(str(optimized_path)) / 1e6 if optimized_path.exists() else orig_size

    print(f'Original ONNX:   {orig_size:.1f} MB')
    print(f'Optimized ONNX:  {opt_size:.1f} MB  ({100*(orig_size-opt_size)/orig_size:.1f}% reduction)')

    # Quick latency comparison: original vs optimized session
    test_input_np = (DUMMY_INPUT if QUICK_MODE else torch.rand(1, 3, IMG_SIZE, IMG_SIZE)).numpy()
    orig_session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    n_bench = 20
    # Warmup
    for _ in range(5):
        orig_session.run(None, {'image': test_input_np})
        opt_session.run(None, {'image': test_input_np})
    orig_times, opt_times = [], []
    for _ in range(n_bench):
        t0 = time.perf_counter(); orig_session.run(None, {'image': test_input_np})
        orig_times.append((time.perf_counter() - t0) * 1000)
        t0 = time.perf_counter(); opt_session.run(None, {'image': test_input_np})
        opt_times.append((time.perf_counter() - t0) * 1000)

    print(f'\nLatency comparison ({n_bench} runs, CPU):')
    print(f'  Original:    median={np.median(orig_times):.1f}ms  P95={np.percentile(orig_times,95):.1f}ms')
    print(f'  Optimized:   median={np.median(opt_times):.1f}ms  P95={np.percentile(opt_times,95):.1f}ms')
    print(f'  Speedup:     {np.median(orig_times)/np.median(opt_times):.2f}x')
else:
    print('[Skipped — ONNX export in cell above must succeed first]')

## 3. TFLite Conversion — For Android / Raspberry Pi

**When you need TFLite:** Android on-device inference (Android NNAPI, GPU delegate), Raspberry Pi (TFLite C API), or MediaPipe pipelines.

**Conversion path:** PyTorch → ONNX → TensorFlow → TFLite (using `onnx-tf`). This is a lossy graph translation — some PyTorch ops don't have exact TF equivalents and require manual kernel mapping.

**Practical note:** For NVIDIA Jetson devices, skip TFLite and use **TensorRT** directly from the ONNX model (`trtexec --onnx model.onnx`) which gives better performance than TFLite on CUDA hardware. TFLite is the right choice specifically for Android and Raspberry Pi.

In [ ]:
# ── ONNX → TFLite conversion (conditional on TF installation) ────────────────
# This cell attempts TFLite conversion via onnx-tf.
# If TensorFlow is not installed, it prints the conversion command instead.

tflite_path = EXPORT_DIR / 'model.tflite'

try:
    import tensorflow as tf
    HAS_TF = True
    print(f'TensorFlow {tf.__version__} found')
except ImportError:
    HAS_TF = False

try:
    import onnx_tf
    HAS_ONNX_TF = True
except ImportError:
    HAS_ONNX_TF = False

if HAS_ONNX and HAS_TF and HAS_ONNX_TF and onnx_path.exists():
    # Step 1: ONNX → SavedModel
    saved_model_dir = EXPORT_DIR / 'tf_saved_model'
    onnx_model = onnx.load(str(onnx_path))
    tf_rep = onnx_tf.backend.prepare(onnx_model)
    tf_rep.export_graph(str(saved_model_dir))

    # Step 2: SavedModel → TFLite FlatBuffer
    converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
    # INT8 post-training quantization for further size reduction
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    tflite_path.write_bytes(tflite_model)

    tflite_size = tflite_path.stat().st_size / 1e6
    print(f'✓ TFLite model saved to {tflite_path}')
    print(f'  Size: {tflite_size:.1f} MB')
else:
    # Print instructions instead of silently skipping
    print('[TFLite conversion skipped — missing dependency]')
    print('\nTo convert manually after installing dependencies:')
    print('  pip install onnx-tf tensorflow')
    print('  python -c "')
    print('    import onnx, onnx_tf, tensorflow as tf')
    print('    model = onnx.load(\'model.onnx\')')
    print('    tf_rep = onnx_tf.backend.prepare(model)')
    print('    tf_rep.export_graph(\'saved_model\')')
    print('    converter = tf.lite.TFLiteConverter.from_saved_model(\'saved_model\')')
    print('    converter.optimizations = [tf.lite.Optimize.DEFAULT]')
    print('    open(\'model.tflite\', \'wb\').write(converter.convert())')
    print('  "')
    print('\nAlternative for Jetson (TensorRT is faster than TFLite on CUDA):')
    print('  trtexec --onnx=model.onnx --saveEngine=model.trt --fp16')

## 4. Inference Benchmarking — Measure What Actually Matters

**Why warmup runs matter:** The first inference calls trigger JIT compilation, CUDA kernel loading, and memory allocation. Including them in your benchmark inflates latency by 5-50x. Always discard at least 10 warmup passes.

**P95 / P99 latency vs median:** Median tells you the typical case. P95 tells you what 1 in 20 frames experiences. For a real-time shelf monitoring system running at 10 FPS, a P99 spike > 100ms means one visible stutter per second. Design for P95 ≤ 50ms.

**Profiling note:** If latency exceeds your budget, use `torch.profiler` to find the bottleneck operation. Common culprits: NMS (non-maximum suppression) in the detection head, FPN upsampling, or data transfer (CPU ↔ GPU).

In [ ]:
# ── Comprehensive inference benchmark: warmup + 100 passes ───────────────────
N_WARMUP = 10
N_BENCH  = 50 if QUICK_MODE else 100

def benchmark_model(session_or_model, input_data, n_warmup=10, n_runs=100,
                    backend='pytorch'):
    """Run warmup passes then benchmark n_runs, returning latency statistics."""
    def run_one(x):
        if backend == 'pytorch':
            with torch.no_grad():
                return session_or_model(x if isinstance(x, list) else [x] if not QUICK_MODE else x)
        else:  # onnxruntime
            return session_or_model.run(None, {'image': x})

    # Warmup — discard these timings
    for _ in range(n_warmup):
        run_one(input_data)

    # Benchmarked runs
    latencies_ms = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        run_one(input_data)
        latencies_ms.append((time.perf_counter() - t0) * 1000)

    return {
        'mean_ms':   np.mean(latencies_ms),
        'median_ms': np.median(latencies_ms),
        'p95_ms':    np.percentile(latencies_ms, 95),
        'p99_ms':    np.percentile(latencies_ms, 99),
        'std_ms':    np.std(latencies_ms),
        'latencies': latencies_ms,
    }

pt_input = DUMMY_INPUT if QUICK_MODE else torch.rand(1, 3, IMG_SIZE, IMG_SIZE)
benchmarks = {}

# Benchmark PyTorch
pt_model = model if QUICK_MODE else model  # same model; named for clarity
benchmarks['PyTorch FP32'] = benchmark_model(
    pt_model, pt_input, n_warmup=N_WARMUP, n_runs=N_BENCH, backend='pytorch')

# Benchmark ONNX Runtime (optimized session)
if HAS_ONNX and onnx_path.exists():
    benchmarks['ONNX Runtime'] = benchmark_model(
        opt_session if 'opt_session' in dir() else orig_session,
        pt_input.numpy(), n_warmup=N_WARMUP, n_runs=N_BENCH, backend='onnxruntime')

print(f'Inference benchmark ({N_BENCH} runs, CPU, {IMG_SIZE}×{IMG_SIZE} input)')
print('─' * 62)
for name, stats in benchmarks.items():
    print(f'{name:25s}  mean={stats["mean_ms"]:6.1f}ms  '
          f'median={stats["median_ms"]:6.1f}ms  '
          f'P95={stats["p95_ms"]:6.1f}ms  '
          f'P99={stats["p99_ms"]:6.1f}ms')
print('─' * 62)

# Latency distribution histogram
if benchmarks:
    fig, axes = plt.subplots(1, len(benchmarks), figsize=(6*len(benchmarks), 4))
    if len(benchmarks) == 1:
        axes = [axes]
    for ax, (name, stats) in zip(axes, benchmarks.items()):
        ax.hist(stats['latencies'], bins=20, color='steelblue', edgecolor='white')
        ax.axvline(stats['median_ms'], color='red',    linestyle='--', label=f'P50={stats["median_ms"]:.1f}ms')
        ax.axvline(stats['p95_ms'],    color='orange', linestyle='--', label=f'P95={stats["p95_ms"]:.1f}ms')
        ax.axvline(50,                 color='lime',   linestyle=':',  label='50ms target')
        ax.set_xlabel('Latency (ms)'); ax.set_ylabel('Count')
        ax.set_title(name)
        ax.legend(fontsize=8)
    plt.suptitle('Inference Latency Distribution (CPU)', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

## 5. Format Comparison Summary

**Decision guide for ProductionCV:**

| Target hardware | Recommended format | Why |
|---|---|---|
| NVIDIA Jetson Nano/Orin | TensorRT (`.trt`) | Best GPU utilisation, FP16/INT8 acceleration |
| Intel NUC / x86 edge server | ONNX Runtime + OpenVINO | CPU/iGPU optimization, wide op support |
| Raspberry Pi 4 | TFLite (INT8) | ARM Cortex-A72 NEON SIMD; TFLite delegate |
| Android | TFLite + NNAPI delegate | Hardware-accelerated on modern Snapdragon |
| Cloud / any | ONNX Runtime | Universal; GPU with CUDA/ROCm providers |

In [ ]:
# ── Format comparison table ───────────────────────────────────────────────────
# Sizes and latencies measured in this notebook; GPU estimates from Jetson benchmarks

import pandas as pd

# Measured values from this session
pt_size  = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6
onnx_size_measured = os.path.getsize(str(onnx_path)) / 1e6 if onnx_path.exists() else float('nan')
pt_lat   = benchmarks.get('PyTorch FP32', {}).get('median_ms', float('nan'))
ort_lat  = benchmarks.get('ONNX Runtime', {}).get('median_ms', float('nan'))

comparison = pd.DataFrame([
    {'Format': 'PyTorch FP32 (.pth)',
     'Size_MB': round(pt_size, 1),
     'Backend': 'PyTorch',
     'Latency_ms_CPU': round(pt_lat, 1),
     'Latency_ms_Jetson': '~42ms (FP32)',
     'Hardware_Target': 'Dev / training'},
    {'Format': 'ONNX FP32 (.onnx)',
     'Size_MB': round(onnx_size_measured, 1),
     'Backend': 'ONNX Runtime',
     'Latency_ms_CPU': round(ort_lat, 1),
     'Latency_ms_Jetson': '~38ms (ORT-CPU)',
     'Hardware_Target': 'x86 edge / cloud'},
    {'Format': 'TensorRT FP16 (.trt)',
     'Size_MB': '~10 (FP16 weights)',
     'Backend': 'TensorRT',
     'Latency_ms_CPU': 'N/A',
     'Latency_ms_Jetson': '~22ms (FP16)',
     'Hardware_Target': 'Jetson Nano/Orin'},
    {'Format': 'TFLite INT8 (.tflite)',
     'Size_MB': '~5 (INT8)',
     'Backend': 'TFLite',
     'Latency_ms_CPU': '~55ms',
     'Latency_ms_Jetson': 'N/A',
     'Hardware_Target': 'Raspberry Pi / Android'},
])

print(comparison.to_string(index=False))
print('\nRecommendation for ProductionCV (<50ms, <100MB): TensorRT FP16 on Jetson Nano')